# 06 PD Model Family Comparison

This notebook compares logistic regression, regularized logistic regression, and tree-based models for Probability of Default modeling.

## 1. Purpose of Milestone 4

Milestone 4 improves the PD modeling workflow by comparing multiple model families on the same LendingClub dataset and the same train-test split.

The goal is not to automatically choose the model with the highest ROC-AUC. The best candidate PD model should balance ranking power, default-class performance, probability calibration, decile separation, time stability, interpretability, and suitability for Expected Loss.

This matters because the selected PD model may later feed Expected Loss:

```text
Expected Loss = PD x LGD x EAD
```

## 2. Import Libraries

In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 3. Load Cleaned Dataset

In [ ]:
data_path = "../data/processed/loans_clean_v1.csv"
tables_dir = Path("../reports/tables")
figures_dir = Path("../reports/figures")

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

loans = pd.read_csv(data_path, low_memory=False)

print(f"Rows: {loans.shape[0]:,}")
print(f"Columns: {loans.shape[1]:,}")

## 4. Define Target and Feature Sets

In [ ]:
target_column = "default_flag"

feature_sets = {
    "Feature Set A": {
        "description": "Full risk/pricing model with grade, sub_grade, and int_rate",
        "numeric": [
            "loan_amnt", "term", "int_rate", "annual_inc", "dti",
            "delinq_2yrs", "revol_util", "total_acc", "open_acc", "emp_length",
        ],
        "categorical": [
            "grade", "sub_grade", "home_ownership", "verification_status",
            "purpose", "addr_state",
        ],
    },
    "Feature Set D": {
        "description": "Borrower-only model without grade, sub_grade, or int_rate",
        "numeric": [
            "loan_amnt", "term", "annual_inc", "dti", "delinq_2yrs",
            "revol_util", "total_acc", "open_acc", "emp_length",
        ],
        "categorical": [
            "home_ownership", "verification_status", "purpose", "addr_state",
        ],
    },
}

required_columns = [target_column]
for feature_set in feature_sets.values():
    required_columns.extend(feature_set["numeric"])
    required_columns.extend(feature_set["categorical"])

missing_columns = [column for column in sorted(set(required_columns)) if column not in loans.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

loans_model = loans.dropna(subset=[target_column]).copy()
loans_model[target_column] = loans_model[target_column].astype(int)

print("All required columns exist.")
print(f"Modeling rows: {len(loans_model):,}")
print(f"Default rate: {loans_model[target_column].mean():.2%}")

## 5. Full Feature Set Versus Borrower-Only Feature Set

Feature Set A includes LendingClub risk and pricing variables: `grade`, `sub_grade`, and `int_rate`. This version is expected to perform strongly because it uses information that already reflects LendingClub's underwriting and pricing assessment.

Feature Set D removes those variables and keeps borrower and loan characteristics only. This version is more independent from LendingClub's internal risk labels, but Milestone 3 showed that it is weaker. Comparing both feature sets helps separate pure predictive performance from business interpretability and independence.

## 6. Create Shared Train-Test Split

In [ ]:
train_index, test_index = train_test_split(
    loans_model.index,
    test_size=0.20,
    random_state=42,
    stratify=loans_model[target_column],
)

print(f"Training rows: {len(train_index):,}")
print(f"Test rows: {len(test_index):,}")
print(f"Training default rate: {loans_model.loc[train_index, target_column].mean():.2%}")
print(f"Test default rate: {loans_model.loc[test_index, target_column].mean():.2%}")

## 7. Why Tree-Based Models May Improve PD Prediction

Logistic regression is transparent and useful as a benchmark, but it assumes a relatively simple relationship between inputs and default risk. Tree-based models can capture nonlinear patterns and interactions, such as the combined effect of high `dti`, high `revol_util`, and longer `term`.

However, tree-based models can be harder to interpret and may produce poorly calibrated probabilities. For credit risk, better ranking is not enough; predicted PDs must also be reasonable probability estimates.

## 8. Optional XGBoost and LightGBM Setup

In [ ]:
xgboost_available = importlib.util.find_spec("xgboost") is not None
lightgbm_available = importlib.util.find_spec("lightgbm") is not None

if xgboost_available:
    from xgboost import XGBClassifier
    print("XGBoost is installed and will be included.")
else:
    print("XGBoost is not installed. It will be skipped.")

if lightgbm_available:
    from lightgbm import LGBMClassifier
    print("LightGBM is installed and will be included.")
else:
    print("LightGBM is not installed. It will be skipped.")

If XGBoost or LightGBM are skipped, they can be installed later with:

```bash
pip install xgboost lightgbm
```

They are optional for this milestone. The notebook will still run with scikit-learn logistic regression, Random Forest, and Gradient Boosting.

## 9. Helper Functions

In [ ]:
def make_one_hot_encoder(dense_output=False):
    """Create OneHotEncoder while supporting older and newer scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=not dense_output)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=not dense_output)


def build_preprocessor(numeric_features, categorical_features, scale_numeric, dense_output=False):
    """Build preprocessing for numeric and categorical features."""
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder(dense_output=dense_output)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
        ]
    )


def build_model_pipeline(model_name, estimator, numeric_features, categorical_features, scale_numeric, dense_output=False):
    """Combine preprocessing and estimator into one pipeline."""
    preprocessor = build_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        scale_numeric=scale_numeric,
        dense_output=dense_output,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )


def create_risk_decile_table(y_true, y_pred_proba, feature_set_name, model_name):
    """Create risk deciles from predicted default probabilities."""
    results = pd.DataFrame({
        "default_flag": y_true.values,
        "predicted_default_probability": y_pred_proba,
    })

    results["decile"] = pd.qcut(
        results["predicted_default_probability"],
        q=10,
        labels=False,
        duplicates="drop",
    ) + 1

    deciles = (
        results.groupby("decile")
        .agg(
            number_of_loans=("default_flag", "size"),
            average_predicted_pd=("predicted_default_probability", "mean"),
            actual_default_rate=("default_flag", "mean"),
            number_of_defaults=("default_flag", "sum"),
            min_predicted_pd=("predicted_default_probability", "min"),
            max_predicted_pd=("predicted_default_probability", "max"),
        )
        .reset_index()
        .sort_values("decile")
    )

    deciles.insert(0, "model_name", model_name)
    deciles.insert(0, "feature_set", feature_set_name)

    return deciles


def calculate_metrics(y_true, y_pred_proba):
    """Calculate ranking, probability, and 0.5-threshold metrics."""
    y_pred_05 = (y_pred_proba >= 0.50).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, y_pred_proba),
        "pr_auc": average_precision_score(y_true, y_pred_proba),
        "brier_score": brier_score_loss(y_true, y_pred_proba),
        "accuracy_at_0_5": accuracy_score(y_true, y_pred_05),
        "precision_at_0_5": precision_score(y_true, y_pred_05, zero_division=0),
        "recall_at_0_5": recall_score(y_true, y_pred_05, zero_division=0),
    }


def get_decile_ratio(decile_table):
    """Calculate highest / lowest decile default-rate ratio."""
    lowest_rate = decile_table.loc[decile_table["decile"].idxmin(), "actual_default_rate"]
    highest_rate = decile_table.loc[decile_table["decile"].idxmax(), "actual_default_rate"]

    return highest_rate / lowest_rate

## 10. Define Model Families

In [ ]:
model_specs = [
    {
        "model_name": "Logistic Regression Baseline",
        "estimator": LogisticRegression(max_iter=1000, random_state=42),
        "scale_numeric": True,
        "dense_output": False,
        "model_family": "linear",
    },
    {
        "model_name": "Regularized Logistic Regression L2",
        "estimator": LogisticRegression(max_iter=1000, penalty="l2", C=0.5, random_state=42),
        "scale_numeric": True,
        "dense_output": False,
        "model_family": "linear",
    },
    {
        "model_name": "Random Forest",
        "estimator": RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=100,
            n_jobs=-1,
            random_state=42,
        ),
        "scale_numeric": False,
        "dense_output": False,
        "model_family": "tree",
    },
    {
        "model_name": "Gradient Boosting",
        "estimator": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
        ),
        "scale_numeric": False,
        "dense_output": True,
        "model_family": "tree",
    },
]

if xgboost_available:
    model_specs.append({
        "model_name": "XGBoost",
        "estimator": XGBClassifier(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.80,
            colsample_bytree=0.80,
            eval_metric="logloss",
            random_state=42,
        ),
        "scale_numeric": False,
        "dense_output": False,
        "model_family": "tree",
    })

if lightgbm_available:
    model_specs.append({
        "model_name": "LightGBM",
        "estimator": LGBMClassifier(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.80,
            colsample_bytree=0.80,
            random_state=42,
        ),
        "scale_numeric": False,
        "dense_output": False,
        "model_family": "tree",
    })

pd.DataFrame([{"model_name": spec["model_name"], "model_family": spec["model_family"]} for spec in model_specs])

## 11. Train and Evaluate Model Families

In [ ]:
comparison_rows = []
all_decile_tables = []
stored_predictions = {}
trained_models = {}
feature_importance_rows = []

for feature_set_name, feature_set in feature_sets.items():
    numeric_features = feature_set["numeric"]
    categorical_features = feature_set["categorical"]
    feature_columns = numeric_features + categorical_features

    X_train = loans_model.loc[train_index, feature_columns]
    X_test = loans_model.loc[test_index, feature_columns]
    y_train = loans_model.loc[train_index, target_column]
    y_test = loans_model.loc[test_index, target_column]

    for spec in model_specs:
        model_name = spec["model_name"]
        print(f"Training {model_name} on {feature_set_name}...")

        model = build_model_pipeline(
            model_name=model_name,
            estimator=clone(spec["estimator"]),
            numeric_features=numeric_features,
            categorical_features=categorical_features,
            scale_numeric=spec["scale_numeric"],
            dense_output=spec["dense_output"],
        )

        model.fit(X_train, y_train)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        metrics = calculate_metrics(y_test, y_pred_proba)
        decile_table = create_risk_decile_table(
            y_true=y_test,
            y_pred_proba=y_pred_proba,
            feature_set_name=feature_set_name,
            model_name=model_name,
        )
        decile_ratio = get_decile_ratio(decile_table)

        comparison_rows.append({
            "feature_set": feature_set_name,
            "feature_set_description": feature_set["description"],
            "model_name": model_name,
            "model_family": spec["model_family"],
            **metrics,
            "highest_lowest_decile_default_rate_ratio": decile_ratio,
        })

        all_decile_tables.append(decile_table)
        stored_predictions[(feature_set_name, model_name)] = {
            "y_test": y_test,
            "y_pred_proba": y_pred_proba,
        }
        trained_models[(feature_set_name, model_name)] = model

        estimator = model.named_steps["model"]
        if hasattr(estimator, "feature_importances_"):
            feature_names = model.named_steps["preprocessor"].get_feature_names_out()
            importances = estimator.feature_importances_
            for feature_name, importance in zip(feature_names, importances):
                feature_importance_rows.append({
                    "feature_set": feature_set_name,
                    "model_name": model_name,
                    "feature": feature_name,
                    "importance": importance,
                })

comparison_table = pd.DataFrame(comparison_rows).sort_values(
    ["roc_auc", "pr_auc", "brier_score"],
    ascending=[False, False, True],
)
decile_table_all = pd.concat(all_decile_tables, ignore_index=True)
feature_importance_table = pd.DataFrame(feature_importance_rows)

comparison_path = tables_dir / "pd_model_family_comparison_v1.csv"
decile_path = tables_dir / "pd_model_family_deciles_v1.csv"
feature_importance_path = tables_dir / "pd_tree_model_feature_importance_v1.csv"

comparison_table.to_csv(comparison_path, index=False)
decile_table_all.to_csv(decile_path, index=False)
feature_importance_table.to_csv(feature_importance_path, index=False)

display(comparison_table)
print(f"Saved model comparison table to: {comparison_path}")
print(f"Saved decile table to: {decile_path}")
print(f"Saved tree feature importance table to: {feature_importance_path}")

## 12. Model Comparison Results

Use the table above to compare model families across ranking, probability quality, and threshold-based metrics. The best candidate model should not be selected by ROC-AUC alone. A good PD model should have strong ROC-AUC, strong PR-AUC, a low Brier Score, clear decile separation, reasonable calibration, stable time-validation performance, and enough interpretability for credit risk use.

## 13. ROC and Precision-Recall Comparison Charts

In [ ]:
plt.figure(figsize=(9, 6))

for (feature_set_name, model_name), prediction_data in stored_predictions.items():
    fpr, tpr, _ = roc_curve(prediction_data["y_test"], prediction_data["y_pred_proba"])
    auc_value = roc_auc_score(prediction_data["y_test"], prediction_data["y_pred_proba"])
    plt.plot(fpr, tpr, label=f"{feature_set_name} - {model_name} ({auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(fontsize=8)
plt.tight_layout()

roc_comparison_path = figures_dir / "pd_model_family_roc_comparison_v1.png"
plt.savefig(roc_comparison_path, dpi=150)
plt.show()

print(f"Saved ROC comparison chart to: {roc_comparison_path}")

In [ ]:
plt.figure(figsize=(9, 6))

for (feature_set_name, model_name), prediction_data in stored_predictions.items():
    precision, recall, _ = precision_recall_curve(
        prediction_data["y_test"],
        prediction_data["y_pred_proba"],
    )
    ap_value = average_precision_score(prediction_data["y_test"], prediction_data["y_pred_proba"])
    plt.plot(recall, precision, label=f"{feature_set_name} - {model_name} ({ap_value:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")
plt.legend(fontsize=8)
plt.tight_layout()

pr_comparison_path = figures_dir / "pd_model_family_pr_comparison_v1.png"
plt.savefig(pr_comparison_path, dpi=150)
plt.show()

print(f"Saved Precision-Recall comparison chart to: {pr_comparison_path}")

## 14. Why Calibration Still Matters

Calibration matters because the model output is a probability. In Expected Loss, PD is multiplied by LGD and EAD. If PD is systematically too high or too low, Expected Loss will also be too high or too low.

Tree-based models may improve ranking but can still produce poorly calibrated probabilities. Always review calibration before choosing a model for credit risk use.

In [ ]:
plt.figure(figsize=(9, 6))

for (feature_set_name, model_name), prediction_data in stored_predictions.items():
    prob_true, prob_pred = calibration_curve(
        prediction_data["y_test"],
        prediction_data["y_pred_proba"],
        n_bins=10,
        strategy="quantile",
    )
    plt.plot(prob_pred, prob_true, marker="o", label=f"{feature_set_name} - {model_name}")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
plt.xlabel("Average Predicted PD")
plt.ylabel("Actual Default Rate")
plt.title("Calibration Curve Comparison")
plt.legend(fontsize=8)
plt.tight_layout()

calibration_comparison_path = figures_dir / "pd_model_family_calibration_comparison_v1.png"
plt.savefig(calibration_comparison_path, dpi=150)
plt.show()

print(f"Saved calibration comparison chart to: {calibration_comparison_path}")

## 15. Risk Decile Comparison

In [ ]:
plt.figure(figsize=(10, 6))

for (feature_set_name, model_name), group in decile_table_all.groupby(["feature_set", "model_name"]):
    label = f"{feature_set_name} - {model_name}"
    plt.plot(group["decile"], group["actual_default_rate"], marker="o", label=label)

plt.xlabel("Predicted Risk Decile")
plt.ylabel("Actual Default Rate")
plt.title("Default Rate by Risk Decile")
plt.xticks(range(1, 11))
plt.legend(fontsize=8)
plt.tight_layout()

decile_comparison_path = figures_dir / "pd_model_family_decile_comparison_v1.png"
plt.savefig(decile_comparison_path, dpi=150)
plt.show()

print(f"Saved decile comparison chart to: {decile_comparison_path}")

A useful PD model should show increasing default rates from low-risk deciles to high-risk deciles. Strong decile separation means the model can rank borrowers in a way that is useful for credit strategy, monitoring, and Expected Loss segmentation.

## 16. Time-Validation Comparison

In [ ]:
time_validation_rows = []

if "issue_d" in loans_model.columns:
    loans_time = loans_model.copy()
    loans_time["issue_d"] = pd.to_datetime(loans_time["issue_d"], format="%b-%Y", errors="coerce")
    loans_time = loans_time.dropna(subset=["issue_d"]).sort_values("issue_d")

    time_train_index = loans_time[loans_time["issue_d"] < "2016-10-01"].index
    time_test_index = loans_time[loans_time["issue_d"] >= "2016-10-01"].index

    for feature_set_name, feature_set in feature_sets.items():
        numeric_features = feature_set["numeric"]
        categorical_features = feature_set["categorical"]
        feature_columns = numeric_features + categorical_features

        X_time_train = loans_time.loc[time_train_index, feature_columns]
        X_time_test = loans_time.loc[time_test_index, feature_columns]
        y_time_train = loans_time.loc[time_train_index, target_column]
        y_time_test = loans_time.loc[time_test_index, target_column]

        for spec in model_specs:
            model_name = spec["model_name"]
            print(f"Time validation: {model_name} on {feature_set_name}...")

            model = build_model_pipeline(
                model_name=model_name,
                estimator=clone(spec["estimator"]),
                numeric_features=numeric_features,
                categorical_features=categorical_features,
                scale_numeric=spec["scale_numeric"],
                dense_output=spec["dense_output"],
            )

            model.fit(X_time_train, y_time_train)
            y_time_pred_proba = model.predict_proba(X_time_test)[:, 1]
            time_metrics = calculate_metrics(y_time_test, y_time_pred_proba)
            time_deciles = create_risk_decile_table(
                y_true=y_time_test,
                y_pred_proba=y_time_pred_proba,
                feature_set_name=feature_set_name,
                model_name=model_name,
            )

            time_validation_rows.append({
                "feature_set": feature_set_name,
                "feature_set_description": feature_set["description"],
                "model_name": model_name,
                "time_train_rows": len(time_train_index),
                "time_test_rows": len(time_test_index),
                "time_test_default_rate": y_time_test.mean(),
                "roc_auc": time_metrics["roc_auc"],
                "pr_auc": time_metrics["pr_auc"],
                "brier_score": time_metrics["brier_score"],
                "highest_lowest_decile_default_rate_ratio": get_decile_ratio(time_deciles),
            })

    time_validation_table = pd.DataFrame(time_validation_rows).sort_values(
        ["roc_auc", "pr_auc", "brier_score"],
        ascending=[False, False, True],
    )
else:
    time_validation_table = pd.DataFrame()
    print("issue_d not found. Time validation requires issue_d in the processed dataset.")

time_validation_path = tables_dir / "pd_model_family_time_validation_v1.csv"
time_validation_table.to_csv(time_validation_path, index=False)

display(time_validation_table)
print(f"Saved time-validation table to: {time_validation_path}")

Time validation tests whether a model trained on older loans still performs on newer loans. This is important because credit risk changes over time due to borrower mix, underwriting rules, and economic conditions. A model that performs well randomly but poorly over time may not be reliable for Expected Loss.

## 17. Feature Importance Interpretation

In [ ]:
if not feature_importance_table.empty:
    top_tree_importance = (
        feature_importance_table
        .sort_values(["model_name", "feature_set", "importance"], ascending=[True, True, False])
        .groupby(["feature_set", "model_name"])
        .head(15)
    )
    display(top_tree_importance)
else:
    print("No tree feature importances were created.")

Feature importance helps explain which variables tree-based models use most. It should be treated as directional evidence, not causal proof. If `grade`, `sub_grade`, or `int_rate` dominate Feature Set A tree models, that confirms the model is relying heavily on LendingClub risk and pricing information.

## 18. Best Candidate PD Model

Do not automatically choose the highest ROC-AUC model. The best candidate PD model should be selected by comparing:

- ROC-AUC
- PR-AUC
- Brier Score
- calibration
- decile separation
- time validation
- interpretability
- suitability for Expected Loss

A tree-based model may be a better candidate if it improves ROC-AUC and PR-AUC without hurting calibration, time stability, or interpretability too much. If a tree model improves ranking but has poor calibration, it may need calibration before feeding Expected Loss.

## 19. Limitations and Next Steps

Main limitations:

- This notebook compares model families, but it does not fully tune hyperparameters.
- Tree-based models may need calibration before being used for Expected Loss.
- Feature Set A may rely heavily on LendingClub's own risk and pricing variables.
- Feature Set D is more independent but likely less predictive.
- LendingClub accepted loans may not represent a full applicant population.
- Time validation is only one stability check and should be expanded later.

Recommended next steps:

- Review the saved comparison tables and charts.
- Decide whether the best candidate model should prioritize performance or independence from LendingClub risk labels.
- Calibrate the selected PD model if needed.
- Save a final selected PD model version.
- Begin LGD and EAD modeling only after the PD model candidate is stable enough.